# FixMatch (baseline) only — CIFAR-10-40

This notebook trains **only `fixmatch`**, so you can run it in its own
Kaggle session in parallel with the other two methods (each in a separate
notebook/session), cutting your total wait time to roughly 1/3.

## Steps
1. **Settings -> Accelerator -> GPU T4x2**.
2. **Run All.**
3. Takes roughly 2 hours for this one method.
4. **When it finishes, click "Save Version"** (top right, near Share) —
   this is what makes your results (and checkpoints, if interrupted)
   permanently saved. Without this, a fresh Kaggle session can wipe
   `/kaggle/working` and you'd lose the results.
5. If interrupted partway, just re-run all cells — it will say `RESUMED
   from checkpoint` and continue, not restart from zero.
6. Once done, come back to this chat with the final printed result for
   `fixmatch` and it'll be combined with the other two methods'
   results into the final 3-way comparison.

In [ ]:
import os
os.makedirs('/kaggle/working/spl_research', exist_ok=True)
os.makedirs('/kaggle/working/checkpoints_fixmatch', exist_ok=True)
print("Folders ready.")

In [ ]:
%%writefile /kaggle/working/spl_research/wideresnet.py
"""
WideResNet-28-2, the exact architecture family used in Sohn et al. (FixMatch),
Zhang et al. (FlexMatch) and Karaliolios et al. (Smooth Pseudo-Labeling) for
CIFAR-10 experiments (table 14 of the SPL paper: depth=28, width=2).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    def __init__(self, in_planes, out_planes, stride, dropRate=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes, momentum=0.001)
        self.relu1 = nn.LeakyReLU(0.1, inplace=True)
        self.conv1 = nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_planes, momentum=0.001)
        self.relu2 = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(out_planes, out_planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.droprate = dropRate
        self.equalInOut = (in_planes == out_planes)
        self.convShortcut = (not self.equalInOut) and nn.Conv2d(
            in_planes, out_planes, kernel_size=1, stride=stride, padding=0, bias=False) or None

    def forward(self, x):
        if not self.equalInOut:
            x = self.relu1(self.bn1(x))
        else:
            out = self.relu1(self.bn1(x))
        out = self.relu2(self.bn2(self.conv1(out if self.equalInOut else x)))
        if self.droprate > 0:
            out = F.dropout(out, p=self.droprate, training=self.training)
        out = self.conv2(out)
        return torch.add(x if self.equalInOut else self.convShortcut(x), out)


class NetworkBlock(nn.Module):
    def __init__(self, nb_layers, in_planes, out_planes, block, stride, dropRate=0.0):
        super().__init__()
        layers = []
        for i in range(int(nb_layers)):
            layers.append(block(in_planes if i == 0 else out_planes, out_planes,
                                 stride if i == 0 else 1, dropRate))
        self.layer = nn.Sequential(*layers)

    def forward(self, x):
        return self.layer(x)


class WideResNet(nn.Module):
    """WRN-28-2 by default, matching table 14 of the SPL paper."""

    def __init__(self, num_classes=10, depth=28, widen_factor=2, dropRate=0.0):
        super().__init__()
        channels = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]
        assert (depth - 4) % 6 == 0
        n = (depth - 4) / 6
        block = BasicBlock
        self.conv1 = nn.Conv2d(3, channels[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.block1 = NetworkBlock(n, channels[0], channels[1], block, 1, dropRate)
        self.block2 = NetworkBlock(n, channels[1], channels[2], block, 2, dropRate)
        self.block3 = NetworkBlock(n, channels[2], channels[3], block, 2, dropRate)
        self.bn1 = nn.BatchNorm2d(channels[3], momentum=0.001)
        self.relu = nn.LeakyReLU(0.1, inplace=True)
        self.fc = nn.Linear(channels[3], num_classes)
        self.channels = channels[3]

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                m.bias.data.zero_()

    def forward(self, x):
        out = self.conv1(x)
        out = self.block1(out)
        out = self.block2(out)
        out = self.block3(out)
        out = self.relu(self.bn1(out))
        out = F.adaptive_avg_pool2d(out, 1)
        out = out.view(-1, self.channels)
        return self.fc(out)


def build_wideresnet(num_classes=10, depth=28, widen_factor=2):
    return WideResNet(num_classes=num_classes, depth=depth, widen_factor=widen_factor)


In [ ]:
%%writefile /kaggle/working/spl_research/data.py
"""
CIFAR-10-40 benchmark, exactly as defined in the paper (§4.1): 4 labeled images
per class, 40 in total, drawn with a fixed random condition. Default seed 2046
(table 14 of the paper).
"""
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler
import torchvision
import torchvision.transforms as T
from PIL import Image

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2471, 0.2435, 0.2616)


class RandAugmentPC(object):
    """Lightweight strong augmentation stand-in for the RandAugment policy used
    by FixMatch/FlexMatch (Cutout is applied separately after ToTensor)."""

    def __init__(self, n=2, m=10):
        self.n = n
        self.m = m
        self.ops = [
            T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
            T.RandomAffine(degrees=15, translate=(0.1, 0.1), shear=10),
            T.RandomPosterize(bits=4, p=0.5),
            T.RandomAutocontrast(p=0.5),
            T.RandomEqualize(p=0.5),
            T.RandomSolarize(threshold=128, p=0.3),
        ]

    def __call__(self, img):
        chosen = np.random.choice(len(self.ops), self.n, replace=False)
        for i in chosen:
            img = self.ops[i](img)
        return img


class Cutout(object):
    def __init__(self, size=16):
        self.size = size

    def __call__(self, img_tensor):
        h, w = img_tensor.shape[1:]
        y = np.random.randint(h)
        x = np.random.randint(w)
        y1, y2 = max(0, y - self.size // 2), min(h, y + self.size // 2)
        x1, x2 = max(0, x - self.size // 2), min(w, x + self.size // 2)
        img_tensor[:, y1:y2, x1:x2] = 0.0
        return img_tensor


def weak_transform():
    return T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(32, padding=4, padding_mode='reflect'),
        T.ToTensor(),
        T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])


def strong_transform():
    return T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomCrop(32, padding=4, padding_mode='reflect'),
        RandAugmentPC(n=2, m=10),
        T.ToTensor(),
        Cutout(size=16),
        T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])


def eval_transform():
    return T.Compose([T.ToTensor(), T.Normalize(CIFAR10_MEAN, CIFAR10_STD)])


class LabeledDataset(Dataset):
    def __init__(self, images, targets, transform):
        self.images, self.targets, self.transform = images, targets, transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])
        return self.transform(img), self.targets[idx]


class UnlabeledDataset(Dataset):
    """Returns (weak_view, strong_view) of the same unlabeled image."""

    def __init__(self, images, weak_t, strong_t):
        self.images, self.weak_t, self.strong_t = images, weak_t, strong_t

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])
        return self.weak_t(img), self.strong_t(img)


def make_cifar10_40_split(root, seed=2046, num_labels=40, num_classes=10):
    """Stratified random split reproducing the paper's CIFAR-10-40 benchmark:
    exactly num_labels/num_classes labeled images per class, rest unlabeled."""
    base = torchvision.datasets.CIFAR10(root=root, train=True, download=True)
    images, targets = base.data, np.array(base.targets)

    rng = np.random.RandomState(seed)
    per_class = num_labels // num_classes
    labeled_idx = []
    for c in range(num_classes):
        idx_c = np.where(targets == c)[0]
        rng.shuffle(idx_c)
        labeled_idx.extend(idx_c[:per_class].tolist())
    labeled_idx = np.array(labeled_idx)
    unlabeled_idx = np.setdiff1d(np.arange(len(targets)), labeled_idx)

    return (images[labeled_idx], targets[labeled_idx],
            images[unlabeled_idx], targets[unlabeled_idx])  # unlabeled targets kept only for monitoring


def build_loaders(root, seed, num_labels, labeled_bs, unlabeled_bs, mu_ratio, eval_bs=256,
                   iters_per_epoch=1024, num_workers=2):
    l_img, l_tgt, u_img, u_tgt = make_cifar10_40_split(root, seed=seed, num_labels=num_labels)

    labeled_set = LabeledDataset(l_img, l_tgt, weak_transform())
    unlabeled_set = UnlabeledDataset(u_img, weak_transform(), strong_transform())

    test_set = torchvision.datasets.CIFAR10(root=root, train=False, download=True)
    test_set = LabeledDataset(test_set.data, np.array(test_set.targets), eval_transform())

    labeled_loader = DataLoader(
        labeled_set, batch_size=labeled_bs, num_workers=num_workers, drop_last=True,
        sampler=RandomSampler(labeled_set, replacement=True, num_samples=iters_per_epoch * labeled_bs))
    unlabeled_loader = DataLoader(
        unlabeled_set, batch_size=unlabeled_bs, num_workers=num_workers, drop_last=True,
        sampler=RandomSampler(unlabeled_set, replacement=True, num_samples=iters_per_epoch * unlabeled_bs))
    test_loader = DataLoader(test_set, batch_size=eval_bs, shuffle=False, num_workers=num_workers)

    return labeled_loader, unlabeled_loader, test_loader, u_tgt


In [ ]:
%%writefile /kaggle/working/spl_research/losses.py
"""
Three unsupervised loss variants, all sharing the same supervised CE term.

1. FixMatch (baseline)         -- Sohn et al. 2020, Eq. 20 of the SPL paper.
   Hard indicator 1(max softmax(weak) > tau), discontinuous derivative.

2. Smooth FixMatch (paper)     -- Karaliolios et al. 2024, Eq. 21.
   Phi(sigma; tau) = ReLU((sigma - tau) / (1 - tau)) replaces the indicator,
   continuous derivative, single global threshold tau, linear shape (mu=1).

3. Adaptive Smooth FixMatch (new, this work)
   Combines two continuous extensions on top of Smooth FixMatch, neither of
   which reintroduces a discontinuity in the loss:
     a) Per-class adaptive threshold tau_c, tracked via an EMA of how often
        each class is confidently predicted on unlabeled data (curriculum
        pseudo-labeling idea from FlexMatch), but tau_c only ever appears
        inside the smooth Phi factor -- never as a hard cutoff. Under-learned
        classes (those FlexMatch would flag as "harder") get a lower,
        easier-to-satisfy threshold, letting the model pull in more of their
        pseudo-labels while still passing through zero smoothly at tau_c.
     b) A progressive shape parameter mu(t): the Phi factor is raised to a
        continuously varying power mu that starts < 1 (concave, more forgiving
        near the threshold, useful while pseudo-labels are still noisy early
        in training) and anneals toward mu >= 1 (sharper, stricter) as
        training progresses and pseudo-labels become more reliable. This
        keeps derivative continuity intact throughout (Phi remains C1 in
        sigma for any mu > 0, and mu itself is a slowly-varying schedule, not
        a function of sigma, so it introduces no new discontinuity in sigma).
"""
import torch
import torch.nn.functional as F


def supervised_ce(logits_x, targets_x):
    return F.cross_entropy(logits_x, targets_x, reduction='mean')


# ---------------------------------------------------------------------------
# 1. FixMatch baseline
# ---------------------------------------------------------------------------
def fixmatch_unsup_loss(logits_u_w, logits_u_s, tau):
    with torch.no_grad():
        probs_u_w = torch.softmax(logits_u_w, dim=-1)
        max_probs, pseudo_labels = torch.max(probs_u_w, dim=-1)
        mask = (max_probs >= tau).float()
    loss = F.cross_entropy(logits_u_s, pseudo_labels, reduction='none') * mask
    return loss.mean(), mask.mean().item()


# ---------------------------------------------------------------------------
# 2. Smooth FixMatch (paper), Eq. 21
# ---------------------------------------------------------------------------
def phi(sigma, tau, mu=1.0, eps=1e-6):
    """Phi(sigma;tau) = ReLU((sigma-tau)/(1-tau)) ** mu , continuous, in [0,1]."""
    base = torch.clamp((sigma - tau) / (1.0 - tau + eps), min=0.0, max=1.0)
    if mu == 1.0:
        return base
    return base.pow(mu)


def smooth_fixmatch_unsup_loss(logits_u_w, logits_u_s, tau, lambda_rescale=1.0):
    with torch.no_grad():
        probs_u_w = torch.softmax(logits_u_w, dim=-1)
        max_probs, pseudo_labels = torch.max(probs_u_w, dim=-1)
        weight = phi(max_probs, tau)  # stop-gradient built in via no_grad block
    loss = F.cross_entropy(logits_u_s, pseudo_labels, reduction='none') * weight
    return lambda_rescale * loss.mean(), weight.mean().item()


# ---------------------------------------------------------------------------
# 3. Adaptive Smooth FixMatch (new)
# ---------------------------------------------------------------------------
class AdaptiveThresholdTracker:
    """Maintains a per-class adaptive threshold tau_c in [tau_min, tau_max],
    based on an EMA of each class's share of confident (>tau_max) predictions
    on unlabeled data -- classes that are confidently predicted less often
    get a lower tau_c, so their (still continuous) Phi factor opens up sooner.
    This is the smoothed, non-hardcoded analogue of FlexMatch's curriculum
    pseudo-labeling, and it never assumes a uniform class prior: tau_c is
    driven purely by the model's own behaviour on the unlabeled set.
    """

    def __init__(self, num_classes, tau_min=0.5, tau_max=0.95, ema_m=0.999, device='cpu'):
        self.num_classes = num_classes
        self.tau_min = tau_min
        self.tau_max = tau_max
        self.ema_m = ema_m
        self.class_conf_count = torch.ones(num_classes, device=device)  # Laplace-smoothed

    @torch.no_grad()
    def update(self, probs_u_w):
        max_probs, pseudo_labels = torch.max(probs_u_w, dim=-1)
        confident = max_probs >= self.tau_max
        if confident.any():
            counts = torch.bincount(pseudo_labels[confident], minlength=self.num_classes).float()
            counts = counts.to(self.class_conf_count.device)
            self.class_conf_count.mul_(self.ema_m).add_(counts, alpha=1 - self.ema_m)

    @torch.no_grad()
    def get_thresholds(self):
        # normalize learning status in [0,1]: 1 = as confident as the best-learned class
        status = self.class_conf_count / self.class_conf_count.max()
        # beta(status) mapping as in FlexMatch's convex curriculum: status/(2-status)
        beta = status / (2.0 - status)
        tau_c = self.tau_min + beta * (self.tau_max - self.tau_min)
        return tau_c  # shape [num_classes]


def mu_schedule(step, total_steps, mu_start=0.6, mu_end=1.4):
    """Progressive shape schedule: concave/forgiving (mu<1) early in training,
    linearly annealed to convex/stricter (mu>1) by the end of training."""
    frac = min(1.0, step / max(1, total_steps))
    return mu_start + frac * (mu_end - mu_start)


def adaptive_smooth_unsup_loss(logits_u_w, logits_u_s, threshold_tracker, step, total_steps,
                                mu_start=0.6, mu_end=1.4):
    with torch.no_grad():
        probs_u_w = torch.softmax(logits_u_w, dim=-1)
        max_probs, pseudo_labels = torch.max(probs_u_w, dim=-1)
        tau_c = threshold_tracker.get_thresholds().to(logits_u_w.device)
        tau_per_sample = tau_c[pseudo_labels]
        mu_t = mu_schedule(step, total_steps, mu_start, mu_end)
        weight = phi(max_probs, tau_per_sample, mu=mu_t)
        threshold_tracker.update(probs_u_w)
    loss = F.cross_entropy(logits_u_s, pseudo_labels, reduction='none') * weight
    return loss.mean(), weight.mean().item(), mu_t


In [ ]:
%%writefile /kaggle/working/spl_research/ema.py
import copy
import math
import torch


class ModelEMA:
    """Exponential moving average of model weights, decay=0.999 as in table 14."""

    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model)
        self.ema.eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        ema_params = dict(self.ema.named_parameters())
        model_params = dict(model.named_parameters())
        for k in ema_params:
            ema_params[k].mul_(self.decay).add_(model_params[k], alpha=1 - self.decay)
        ema_buffers = dict(self.ema.named_buffers())
        model_buffers = dict(model.named_buffers())
        for k in ema_buffers:
            ema_buffers[k].copy_(model_buffers[k])


def cosine_lr_lambda(total_steps):
    # FixMatch-style cosine schedule: lr * cos(7*pi*step / (16*total_steps))
    def f(step):
        return max(0.0, math.cos(7 * math.pi * step / (16 * total_steps)))
    return f


In [ ]:
%%writefile /kaggle/working/spl_research/run_comparison.py
"""
Reproduces the paper's CIFAR-10-40 benchmark for three methods and compares them:

  1. FixMatch                  (baseline, Sohn et al. 2020)
  2. Smooth FixMatch            (Karaliolios et al. 2024 -- the paper's contribution)
  3. Adaptive Smooth FixMatch   (this work -- per-class adaptive threshold +
                                 progressive smoothing shape on top of Smooth FixMatch)

Usage (defaults are a *fast, reduced-scale* smoke-test config so it finishes in
minutes on a laptop CPU/GPU; for a paper-scale run on Kaggle, override with the
--iters-per-epoch / --epochs flags to approach the paper's 2**20 total iterations,
see the "SCALING UP" note at the bottom of this file).

Example (Kaggle, near paper-scale):
    python run_comparison.py --epochs 1024 --iters-per-epoch 1024 \
        --labeled-bs 64 --unlabeled-bs 448 --data-root /kaggle/working/data

Example (quick smoke test, what runs by default):
    python run_comparison.py
"""
import argparse
import ctypes
import gc
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD
from torch.optim.lr_scheduler import LambdaLR

from wideresnet import build_wideresnet
from data import build_loaders
from ema import ModelEMA, cosine_lr_lambda
from losses import (supervised_ce, fixmatch_unsup_loss, smooth_fixmatch_unsup_loss,
                     adaptive_smooth_unsup_loss, AdaptiveThresholdTracker)


def free_memory():
    """Release both CPU RAM (Python/PIL objects, per the malloc_trim fix that
    was needed for the gender-classification pipeline's PIL leak) and cached
    but unused GPU memory (PyTorch's allocator holds onto freed GPU blocks by
    default -- empty_cache() hands them back to the OS/driver so a long run
    with many epochs doesn't slowly climb toward OOM)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except (OSError, AttributeError):
        pass  # not on Linux / libc unavailable -- safe to skip


def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        pred = logits.argmax(dim=-1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    model.train()
    return 100.0 * correct / total


def train_one_method(method, args, device):
    print(f"\n{'=' * 70}\nTraining method: {method}\n{'=' * 70}")
    set_seed(args.seed)

    labeled_loader, unlabeled_loader, test_loader, _ = build_loaders(
        root=args.data_root, seed=args.seed, num_labels=args.num_labels,
        labeled_bs=args.labeled_bs, unlabeled_bs=args.unlabeled_bs, mu_ratio=args.unlabeled_bs / args.labeled_bs,
        iters_per_epoch=args.iters_per_epoch, num_workers=args.num_workers)

    # base_model is the "real" module: EMA and the optimizer always refer to
    # this one, so their parameter names stay stable regardless of whether we
    # wrap it in DataParallel below for multi-GPU forward/backward.
    base_model = build_wideresnet(num_classes=args.num_classes, depth=args.depth,
                                   widen_factor=args.widen_factor).to(device)

    n_gpus = torch.cuda.device_count()
    if n_gpus > 1:
        print(f"Using {n_gpus} GPUs via DataParallel (e.g. T4x2).")
        model = nn.DataParallel(base_model)
    else:
        model = base_model

    ema = ModelEMA(base_model, decay=args.ema_decay)

    no_decay = ['bn', 'bias']
    grouped = [
        {'params': [p for n, p in base_model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': args.weight_decay},
        {'params': [p for n, p in base_model.named_parameters() if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optimizer = SGD(grouped, lr=args.lr, momentum=args.momentum, nesterov=True)
    total_steps = args.epochs * args.iters_per_epoch
    scheduler = LambdaLR(optimizer, lr_lambda=cosine_lr_lambda(total_steps))

    threshold_tracker = None
    if method == 'adaptive_smooth':
        threshold_tracker = AdaptiveThresholdTracker(
            num_classes=args.num_classes, tau_min=args.tau_min, tau_max=args.tau,
            ema_m=0.999, device=device)

    labeled_iter = iter(labeled_loader)
    unlabeled_iter = iter(unlabeled_loader)

    history = {'step': [], 'test_acc_raw': [], 'test_acc_ema': [], 'mask_rate': []}
    step = 0
    start_epoch = 0
    mask_rate = 0.0

    # ---- RESUME: if a checkpoint for this method already exists, load it
    # instead of starting from scratch. This is what protects you from
    # losing hours of progress to a dropped connection / interrupted session.
    ckpt_path = os.path.join(args.checkpoint_dir, f"{method}_ckpt.pt")
    if args.resume and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        base_model.load_state_dict(ckpt['base_model'])
        ema.ema.load_state_dict(ckpt['ema_model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        history = ckpt['history']
        step = ckpt['step']
        start_epoch = ckpt['epoch'] + 1
        if threshold_tracker is not None and ckpt.get('threshold_tracker') is not None:
            threshold_tracker.class_conf_count = ckpt['threshold_tracker'].to(device)
        print(f"[{method}] RESUMED from checkpoint at epoch {start_epoch}/{args.epochs} "
              f"(step {step}/{total_steps}) -- picking up where the interrupted run left off.")

    t0 = time.time()
    for epoch in range(start_epoch, args.epochs):
        for _ in range(args.iters_per_epoch):
            try:
                x_l, y_l = next(labeled_iter)
            except StopIteration:
                labeled_iter = iter(labeled_loader)
                x_l, y_l = next(labeled_iter)
            try:
                (x_uw, x_us) = next(unlabeled_iter)
            except StopIteration:
                unlabeled_iter = iter(unlabeled_loader)
                (x_uw, x_us) = next(unlabeled_iter)

            x_l, y_l = x_l.to(device), y_l.to(device)
            x_uw, x_us = x_uw.to(device), x_us.to(device)

            inputs = torch.cat([x_l, x_uw, x_us], dim=0)
            logits = model(inputs)
            logits_x = logits[:x_l.size(0)]
            logits_u_w, logits_u_s = logits[x_l.size(0):].chunk(2)

            sup_loss = supervised_ce(logits_x, y_l)

            if method == 'fixmatch':
                unsup_loss, mask_rate = fixmatch_unsup_loss(logits_u_w, logits_u_s, tau=args.tau)
            elif method == 'smooth_fixmatch':
                unsup_loss, mask_rate = smooth_fixmatch_unsup_loss(
                    logits_u_w, logits_u_s, tau=args.tau, lambda_rescale=args.lambda_smooth_rescale)
            elif method == 'adaptive_smooth':
                unsup_loss, mask_rate, mu_t = adaptive_smooth_unsup_loss(
                    logits_u_w, logits_u_s, threshold_tracker, step, total_steps,
                    mu_start=args.mu_start, mu_end=args.mu_end)
            else:
                raise ValueError(method)

            loss = sup_loss + args.lambda_u * unsup_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            ema.update(base_model)
            step += 1

        # Release CPU+GPU memory once per epoch. Cheap relative to an epoch's
        # ~500 iterations, and prevents the slow OOM-by-a-thousand-cuts that
        # hit the gender-classification pipeline (PIL objects from the many
        # weak/strong augmentation calls this epoch, plus PyTorch's cached
        # allocator blocks that would otherwise just sit there unused).
        free_memory()

        if (epoch + 1) % args.eval_every == 0 or epoch == args.epochs - 1:
            acc_raw = evaluate(base_model, test_loader, device)
            acc_ema = evaluate(ema.ema, test_loader, device)
            history['step'].append(step)
            history['test_acc_raw'].append(acc_raw)
            history['test_acc_ema'].append(acc_ema)
            history['mask_rate'].append(mask_rate)
            elapsed = time.time() - t0
            print(f"[{method}] epoch {epoch + 1}/{args.epochs} step {step}/{total_steps} "
                  f"acc(raw)={acc_raw:.2f}% acc(ema)={acc_ema:.2f}% mask_rate={mask_rate:.3f} "
                  f"({elapsed:.1f}s)")

            # ---- CHECKPOINT: save right after every eval, so a dropped
            # connection never costs you more than `eval_every` epochs.
            os.makedirs(args.checkpoint_dir, exist_ok=True)
            torch.save({
                'epoch': epoch,
                'step': step,
                'base_model': base_model.state_dict(),
                'ema_model': ema.ema.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'history': history,
                'threshold_tracker': threshold_tracker.class_conf_count.cpu()
                if threshold_tracker is not None else None,
            }, ckpt_path)
            print(f"[{method}] checkpoint saved -> {ckpt_path}")

    final_acc = history['test_acc_ema'][-1]
    final_err = 100.0 - final_acc
    result = {'method': method, 'final_test_acc': final_acc, 'final_error_rate': final_err,
              'history': history}

    # Explicitly tear down this method's model/optimizer/loaders before the
    # next method starts (you run all 3 back-to-back in one process/session)
    # so GPU memory from this run doesn't linger into the next one.
    del model, base_model, ema, optimizer, scheduler, labeled_loader, unlabeled_loader
    del labeled_iter, unlabeled_iter, test_loader
    if threshold_tracker is not None:
        del threshold_tracker
    free_memory()

    return result


def build_arg_parser():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-root', type=str, default='/home/claude/spl_research/cifar_data')
    parser.add_argument('--seed', type=int, default=2046)  # paper's default seed, table 14
    parser.add_argument('--num-labels', type=int, default=40)  # CIFAR-10-40 benchmark
    parser.add_argument('--num-classes', type=int, default=10)
    parser.add_argument('--depth', type=int, default=28)
    parser.add_argument('--widen-factor', type=int, default=2)
    parser.add_argument('--labeled-bs', type=int, default=8)     # paper: 64
    parser.add_argument('--unlabeled-bs', type=int, default=16)  # paper: 448 (7x labeled)
    parser.add_argument('--lr', type=float, default=0.03)
    parser.add_argument('--momentum', type=float, default=0.9)
    parser.add_argument('--weight-decay', type=float, default=5e-4)
    parser.add_argument('--ema-decay', type=float, default=0.999)
    parser.add_argument('--tau', type=float, default=0.95)
    parser.add_argument('--tau-min', type=float, default=0.5)
    parser.add_argument('--lambda-u', type=float, default=1.0)
    parser.add_argument('--lambda-smooth-rescale', type=float, default=1.1)  # per §C.1, ~1.1 for CIFAR
    parser.add_argument('--mu-start', type=float, default=0.6)
    parser.add_argument('--mu-end', type=float, default=1.4)
    parser.add_argument('--epochs', type=int, default=4)          # paper: up to 1024 (2**20 iters)
    parser.add_argument('--iters-per-epoch', type=int, default=32)  # paper: 1024
    parser.add_argument('--eval-every', type=int, default=1)
    parser.add_argument('--num-workers', type=int, default=2)
    parser.add_argument('--methods', type=str, nargs='+',
                         default=['fixmatch', 'smooth_fixmatch', 'adaptive_smooth'])
    parser.add_argument('--out', type=str, default='/home/claude/spl_research/results.json')
    parser.add_argument('--checkpoint-dir', type=str,
                         default='/home/claude/spl_research/checkpoints')
    parser.add_argument('--resume', action='store_true', default=True,
                         help='Resume each method from its last checkpoint if one exists.')
    return parser


def get_args(**overrides):
    """Build an args object WITHOUT touching sys.argv. Safe to call from any
    notebook kernel (Colab, Kaggle, plain Jupyter) since it never parses the
    process's real command line, which is polluted by kernel-launcher flags
    like '-f /root/.../kernel-xxxx.json' in those environments."""
    parser = build_arg_parser()
    args = parser.parse_args([])  # parse an empty list -> just fills in defaults
    for k, v in overrides.items():
        key = k.replace('-', '_')
        if not hasattr(args, key):
            raise ValueError(f"Unknown argument: {k}")
        setattr(args, key, v)
    return args


def main(args=None):
    if args is None:
        # Only touch sys.argv when actually run as a script from a real
        # command line (e.g. `python run_comparison.py --epochs 10`).
        args = build_arg_parser().parse_args()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    results = []
    for method in args.methods:
        res = train_one_method(method, args, device)
        results.append(res)

    print(f"\n{'=' * 70}\nFINAL COMPARISON (CIFAR-10-{args.num_labels}, seed {args.seed})\n{'=' * 70}")
    print(f"{'Method':<22}{'Test Acc (%)':<16}{'Error Rate (%)':<16}{'Gain vs FixMatch':<18}")
    baseline_err = None
    for r in results:
        if r['method'] == 'fixmatch':
            baseline_err = r['final_error_rate']
    for r in results:
        gain = '' if baseline_err is None else f"{baseline_err - r['final_error_rate']:+.2f}"
        print(f"{r['method']:<22}{r['final_test_acc']:<16.2f}{r['final_error_rate']:<16.2f}{gain:<18}")

    with open(args.out, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved detailed results to {args.out}")


if __name__ == '__main__':
    main()


# -----------------------------------------------------------------------------
# SCALING UP TO PAPER-SCALE RUNS ON KAGGLE
# -----------------------------------------------------------------------------
# The paper trains for 2**20 iterations (~1M) with labeled_bs=64, unlabeled_bs=448,
# which is impractical to smoke-test here. For a real run on your Kaggle P100/T4x2:
#
#   python run_comparison.py \
#       --epochs 1024 --iters-per-epoch 1024 \
#       --labeled-bs 64 --unlabeled-bs 448 \
#       --lr 0.03 --momentum 0.9 --weight-decay 5e-4 --ema-decay 0.999 \
#       --tau 0.95 --eval-every 10 \
#       --data-root /kaggle/working/data
#
# This exactly reproduces the paper's table 14 hyperparameters. Expect run time
# to be many hours on a single P100 (the paper reports similar). Consider:
#   - checkpointing every --eval-every epochs (add torch.save around the eval
#     block) so you can resume across Kaggle's 12h/9h session limits,
#   - running the three methods in three separate Kaggle sessions in parallel
#     rather than sequentially in one script, given your OOM history with
#     paper_improved_kaggle.py -- keep unlabeled_bs lower (e.g. 224) if you hit
#     OOM at epoch transitions, and it will not break comparability since all
#     three methods use the same reduced batch size.


## Step 1 — Smoke test (fast, just confirms it runs)

In [ ]:
import sys
sys.path.append('/kaggle/working/spl_research')

import importlib
import run_comparison
importlib.reload(run_comparison)

smoke_args = run_comparison.get_args(
    methods=['fixmatch'],
    epochs=2,
    iters_per_epoch=10,
    labeled_bs=8,
    unlabeled_bs=16,
    eval_every=1,
    num_workers=0,
    data_root='/kaggle/working/data',
    out='/kaggle/working/results_smoketest_fixmatch.json',
    checkpoint_dir='/kaggle/working/checkpoints_smoketest_fixmatch',
)
run_comparison.main(smoke_args)
print("\nSmoke test finished with no errors -> safe to run the real training below.")

## Step 2 — Real run: fixmatch only

~2 hours. `num_workers=0` is used to avoid the overnight RAM creep seen
before. If you hit an OOM error, lower `unlabeled_bs` to 192 or 96 and
re-run this cell — resume will pick up from the last checkpoint either way.

In [ ]:
real_args = run_comparison.get_args(
    methods=['fixmatch'],
    epochs=60,
    iters_per_epoch=500,       # 60 * 500 = 30,000 steps
    labeled_bs=64,
    unlabeled_bs=256,
    lr=0.03,
    momentum=0.9,
    weight_decay=5e-4,
    ema_decay=0.999,
    tau=0.95,
    eval_every=5,
    num_workers=0,
    data_root='/kaggle/working/data',
    out='/kaggle/working/results_fixmatch.json',
    checkpoint_dir='/kaggle/working/checkpoints_fixmatch',
    resume=True,
)
run_comparison.main(real_args)

## Done? 

Click **"Save Version"** now (top right) before closing this session,
so your result is permanently saved. Then copy the printed comparison
table (or `results_fixmatch.json`) back into the main thread.